In [22]:
# https://github.com/jeandeducla/ML-Time-Series/blob/master/SVM-Accelerometer.ipynb
import pandas as pd
import numpy as np
import datetime as dt

In [2]:
import scipy.stats as st
from scipy.fftpack import fft, fftfreq 
from scipy.signal import argrelextrema
import operator

In [3]:
df=pd.read_csv('database_gas.csv')

In [4]:
df.head()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
0,2019-11-06 11:37:13.038174+01:00,19.48,54.86,128.0,563.0,476.0,731.0,649.0,565.0,128.0
1,2019-11-06 11:37:32.744996+01:00,19.59,54.23,129.0,563.0,477.0,731.0,649.0,565.0,125.0
2,2019-11-06 11:37:53.018087+01:00,19.63,54.05,128.0,566.0,478.0,732.0,649.0,565.0,125.0
3,2019-11-06 11:38:13.093151+01:00,19.64,53.74,128.0,566.0,478.0,732.0,649.0,565.0,125.0
4,2019-11-06 11:38:33.032832+01:00,19.67,53.53,128.0,569.0,480.0,732.0,650.0,565.0,125.0


In [5]:
# need to convert time string to time format for easy processing
df['recordtime'] = df['timestamp']
df.recordtime =df.recordtime.apply(lambda x: pd.to_datetime(x).strftime("%Y-%m-%d %H:%M:%S"))
df.recordtime =df.recordtime.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

In [6]:
# sort values from small time to large time
# how much signal is difference from the prevous time to current time is important
df.sort_values(by='recordtime', inplace=True, ascending=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp'] = df['timestamp'].dt.floor(freq='4H')

In [7]:
df.head()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue,recordtime
0,2019-11-06 08:00:00+01:00,19.48,54.86,128.0,563.0,476.0,731.0,649.0,565.0,128.0,2019-11-06 11:37:13
1,2019-11-06 08:00:00+01:00,19.59,54.23,129.0,563.0,477.0,731.0,649.0,565.0,125.0,2019-11-06 11:37:32
2,2019-11-06 08:00:00+01:00,19.63,54.05,128.0,566.0,478.0,732.0,649.0,565.0,125.0,2019-11-06 11:37:53
3,2019-11-06 08:00:00+01:00,19.64,53.74,128.0,566.0,478.0,732.0,649.0,565.0,125.0,2019-11-06 11:38:13
4,2019-11-06 08:00:00+01:00,19.67,53.53,128.0,569.0,480.0,732.0,650.0,565.0,125.0,2019-11-06 11:38:33


In [8]:
df.tail()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue,recordtime
416148,2020-02-13 08:00:00+01:00,20.62,57.75,62.0,518.0,524.0,698.0,642.0,591.0,157.0,2020-02-13 11:56:53
416149,2020-02-13 08:00:00+01:00,20.62,57.75,62.0,518.0,524.0,698.0,641.0,591.0,157.0,2020-02-13 11:57:14
416150,2020-02-13 08:00:00+01:00,20.62,57.75,62.0,518.0,524.0,698.0,641.0,591.0,157.0,2020-02-13 11:57:33
416151,2020-02-13 08:00:00+01:00,20.62,57.74,63.0,518.0,524.0,698.0,641.0,591.0,157.0,2020-02-13 11:57:54
416152,2020-02-13 08:00:00+01:00,20.62,57.74,63.0,518.0,524.0,698.0,641.0,591.0,157.0,2020-02-13 11:58:14


In [9]:
df.shape

(416153, 11)

In [10]:
# look the count for each time group
group_count = pd.DataFrame(df.groupby('timestamp').count()['humidity'])
group_count.reset_index(inplace=True)
group_count.rename(columns={"humidity": "countv"}, inplace=True)

In [11]:
# some of the time group has low number of data missing
# Fillup them
group_count.countv.unique()

array([ 69, 720, 719, 721, 717, 710, 711,   5,  83, 718, 712, 702, 715,
       642, 695, 608, 684, 688, 407, 634, 603])

In [12]:
# these time group has low count - 
# lot of data is missing for these. So they need to be removed
time_low_count = group_count[group_count.countv<150]
time_low_count

,timestamp,countv
0,2019-11-06 08:00:00+01:00,69
195,2019-12-08 20:00:00+01:00,5
196,2019-12-09 08:00:00+01:00,83
527,2020-02-02 12:00:00+01:00,407


In [13]:
df.columns

Index(['timestamp', 'temperature', 'humidity', 'CO2CosIRValue',
       'CO2MG811Value', 'MOX1', 'MOX2', 'MOX3', 'MOX4', 'COValue',
       'recordtime'],
      dtype='object')

In [14]:
# investigate how these data is missing
processed_df = df[['temperature', 'humidity', 'CO2CosIRValue','CO2MG811Value', 'MOX1', 'MOX2', 'MOX3', 'MOX4', 'COValue']].copy()
processed_df = processed_df.rolling(3).mean()
processed_df['timestamp'] = df['timestamp']
processed_df['recordtime'] = df['recordtime']

In [15]:
processed_df=processed_df[~processed_df.timestamp.isin(time_low_count.timestamp)]

In [16]:
processed_df.to_csv('processed_df.csv')

In [17]:
group_count = group_count[~group_count.timestamp.isin(time_low_count.timestamp)]

In [18]:
group_count.to_csv('group_count.csv')

In [21]:
processed_df.head(10)

,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue,timestamp,recordtime
69,19.793333,48.096667,86.666667,610.333333,550.666667,777.333333,728.000000,648.000000,103.666667,2019-11-06 12:00:00+01:00,2019-11-06 12:00:12
70,19.826667,47.996667,86.000000,611.000000,552.333333,777.666667,729.000000,649.666667,103.000000,2019-11-06 12:00:00+01:00,2019-11-06 12:00:33
71,19.860000,47.893333,85.000000,612.000000,553.666667,778.333333,730.000000,650.333333,102.333333,2019-11-06 12:00:00+01:00,2019-11-06 12:00:53
72,19.860000,47.753333,84.333333,612.666667,554.000000,778.666667,731.000000,651.000000,102.000000,2019-11-06 12:00:00+01:00,2019-11-06 12:01:13
73,19.853333,47.680000,83.666667,613.000000,554.000000,779.000000,732.333333,651.666667,102.000000,2019-11-06 12:00:00+01:00,2019-11-06 12:01:33
74,19.820000,47.680000,83.333333,613.333333,554.666667,779.666667,734.333333,653.000000,102.000000,2019-11-06 12:00:00+01:00,2019-11-06 12:01:53
75,19.786667,47.820000,83.000000,613.666667,555.333333,780.333333,736.000000,654.666667,101.666667,2019-11-06 12:00:00+01:00,2019-11-06 12:02:12
76,19.760000,47.880000,82.666667,614.000000,556.333333,781.000000,737.333333,656.000000,101.333333,2019-11-06 12:00:00+01:00,2019-11-06 12:02:32
77,19.760000,47.813333,82.333333,614.000000,557.000000,781.333333,738.000000,657.000000,101.000000,2019-11-06 12:00:00+01:00,2019-11-06 12:02:52
78,19.786667,47.583333,82.000000,614.000000,557.666667,781.666667,738.666667,657.666667,101.000000,2019-11-06 12:00:00+01:00,2019-11-06 12:03:13


In [26]:
result=processed_df.groupby('timestamp').count()['humidity']

In [25]:
result.unique()

array([720, 719, 721, 717, 710, 711, 718, 712, 702, 715, 642, 695, 608,
       684, 688, 634, 603])

In [ ]:
columns_to_filter = processed_df.columns

# Initialize Kalman filter
kf = KalmanFilter(initial_state_mean=0, n_dim_obs=1)

# Apply Kalman filter to each column
for col in columns_to_filter:
    # Filter the non-null values in the column
    non_null_values = processed_df[col].dropna()
    (filtered_state_means, _) = kf.filter(non_null_values)
    
    # Replace the original column with filtered values
    processed_df[col] = pd.Series(filtered_state_means.flatten(), index=non_null_values.index)
